# 05 - Interpretability and Permutation Importance

## Aim of this notebook

Notebook 4 trained machine-learning models to predict a physics-informed transport-related descriptor from hydrogen-defect descriptors.

In this notebook, we interpret the trained models using feature-importance analysis.

We compare:

1. Random Forest impurity-based feature importance,
2. permutation importance on the test set,
3. optional SHAP analysis if available.

The goal is to identify which defect, spectral, and localization descriptors most strongly influence the model predictions.

Because the transport descriptor is analytically defined using mean IPR and defect density, we interpret the full-feature and reduced-feature models separately.


## Mathematical Formulas Used in This Notebook

This notebook asks: which input features are most important for predicting the transport descriptor?

For a trained model $f$, the prediction is

$$
\hat{y}_i = f(\mathbf{x}_i).
$$

The baseline model score is computed on the unmodified test set:

$$
S_{\mathrm{base}} = S(y, \hat{y}).
$$

For permutation importance, one feature $x_j$ is randomly shuffled across test samples. This breaks the relationship between that feature and the target while keeping the feature distribution unchanged.

The score after shuffling feature $j$ is

$$
S_{\mathrm{perm},j} = S\left(y, f(\mathbf{x}_{\pi(j)})\right),
$$

where $\pi(j)$ means that only feature $j$ has been permuted.

The permutation importance is

$$
I_j = S_{\mathrm{base}} - S_{\mathrm{perm},j}.
$$

If $I_j$ is large, the model depends strongly on feature $j$. If $I_j \approx 0$, the feature contributes little to predictive performance.

For repeated permutations, the reported mean importance is

$$
\overline{I}_j = \frac{1}{R}\sum_{r=1}^{R} I_{j,r},
$$

and the uncertainty is measured using the standard deviation

$$
\sigma_j = \sqrt{\frac{1}{R-1}\sum_{r=1}^{R}(I_{j,r}-\overline{I}_j)^2}.
$$


## How to read this notebook

This notebook explains the trained model instead of only reporting prediction scores.

It focuses on two types of importance analysis:

1. **Random Forest built-in feature importance**: how often and how strongly a feature is used inside the forest.
2. **Permutation importance**: how much model performance decreases when one feature is randomly shuffled.

Permutation importance is especially useful because it asks a more scientific question:  
“If this feature were destroyed, how much would the prediction quality suffer?”


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

### Mathematical meaning of permutation importance

For feature $j$, the importance is the decrease in model score after shuffling that feature:

$$
I_j = S_{\mathrm{base}} - S_{\mathrm{perm},j}.
$$

If shuffling feature $j$ strongly damages prediction quality, then $I_j$ becomes large.


### Explanation of the code cell above

This cell imports the tools needed for interpretability.

The key new tool is `permutation_importance` from scikit-learn. It tests feature importance by randomly shuffling each feature and measuring how much the model performance decreases.

This notebook focuses on understanding model behavior, not only achieving high accuracy.


## Load Dataset and Prepare Features

In [ ]:
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
FIGURE_DIR = ROOT / 'figures'
RESULTS_DIR = ROOT / 'results'

FIGURE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

dataset = pd.read_csv(DATA_DIR / 'hydrogen_defect_dataset.csv')

dataset_ml = dataset.copy()
dataset_ml['clustering_index'] = dataset_ml['clustering_index'].fillna(0.0)

dataset_ml.head()

### Explanation of the code cell above

This cell loads the dataset and prepares it in the same way as Notebook 4.

The missing `clustering_index` values are filled with `0.0` so that the Random Forest model can use the feature matrix without missing entries.

Keeping preprocessing consistent is important. Otherwise, model results from different notebooks would not be comparable.


## Define Feature Groups

The full-feature model includes `defect_density` and `mean_ipr`, which directly define the target.

The reduced-feature model removes these direct variables and tests whether the transport proxy can be inferred from more indirect descriptors.

In [ ]:
target = 'transport_descriptor'

full_features = [
    'defect_density',
    'defect_strength',
    'mean_defect_distance',
    'clustering_index',
    'energy_gap',
    'energy_bandwidth',
    'mean_ipr',
    'max_ipr',
]

reduced_features = [
    'defect_strength',
    'mean_defect_distance',
    'clustering_index',
    'energy_gap',
    'energy_bandwidth',
    'max_ipr',
]

### Explanation of the code cell above

This cell defines the target and two feature groups.

- `full_features` contains all selected descriptors.
- `reduced_features` contains a smaller interpretable subset.
- `target` is the transport descriptor.

Using both feature groups allows us to compare whether interpretability conclusions change when the model has fewer input variables.


## Train Interpretable Baseline Models Again

For interpretability, we retrain Random Forest models using the same feature groups from Notebook 4. Random Forest is useful here because it supports impurity-based feature importance and works naturally with permutation importance.

In [ ]:
def train_random_forest_for_interpretability(features, dataset_ml, target):
    X = dataset_ml[features]
    y = dataset_ml[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=1,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        'model': model,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'y_pred': y_pred,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred),
    }


full_rf_results = train_random_forest_for_interpretability(
    full_features,
    dataset_ml,
    target,
)

reduced_rf_results = train_random_forest_for_interpretability(
    reduced_features,
    dataset_ml,
    target,
)

model_score_table = pd.DataFrame(
    [
        {
            'experiment': 'Full-feature Random Forest',
            'MAE': full_rf_results['MAE'],
            'RMSE': full_rf_results['RMSE'],
            'R2': full_rf_results['R2'],
        },
        {
            'experiment': 'Reduced-feature Random Forest',
            'MAE': reduced_rf_results['MAE'],
            'RMSE': reduced_rf_results['RMSE'],
            'R2': reduced_rf_results['R2'],
        },
    ]
)

model_score_table.to_csv(RESULTS_DIR / 'notebook5_interpretability_model_scores.csv', index=False)
model_score_table

### Explanation of the code cell above

This cell defines a function to train a Random Forest model for interpretability.

The function splits the data, trains the model, predicts on the test set, and returns both the fitted model and evaluation information.

Random Forest is useful here because it provides built-in feature importance and works well for nonlinear relationships.


## Random Forest Feature Importance

Random Forest feature importance measures how much each feature helps reduce prediction error inside the trees.

In [ ]:
def get_rf_feature_importance(model, features):
    return pd.DataFrame(
        {
            'feature': features,
            'rf_importance': model.feature_importances_,
        }
    ).sort_values('rf_importance', ascending=False)


full_rf_importance = get_rf_feature_importance(
    full_rf_results['model'],
    full_features,
)

reduced_rf_importance = get_rf_feature_importance(
    reduced_rf_results['model'],
    reduced_features,
)

full_rf_importance.to_csv(RESULTS_DIR / 'notebook5_rf_importance_full.csv', index=False)
reduced_rf_importance.to_csv(RESULTS_DIR / 'notebook5_rf_importance_reduced.csv', index=False)

full_rf_importance

### Mathematical meaning of Random Forest feature importance

For tree-based models, feature importance is often estimated from how much each feature reduces impurity across the forest:

$$
FI_j = \frac{1}{M}\sum_{m=1}^{M}\sum_{s\in S_{m,j}} \Delta \mathcal{I}_{s},
$$

where $M$ is the number of trees and $\Delta \mathcal{I}_{s}$ is the impurity reduction at split $s$ using feature $j$.


### Explanation of the code cell above

This cell computes built-in Random Forest feature importance.

The importance values come from how much each feature reduces prediction error inside the trees.

The results are saved to CSV files so they can be used in reports or compared with permutation importance.


In [ ]:
reduced_rf_importance

### Explanation of the code cell above

This cell displays the reduced-model Random Forest importance table.

It is useful to inspect the reduced case separately because a smaller feature set often gives a clearer scientific story.

Features with larger importance values are used more strongly by the Random Forest.


## Permutation Importance

Permutation importance is the main new analysis in Notebook 5.

Random Forest feature importance can be biased when features are correlated. Permutation importance is more reliable because it asks:

> If I randomly shuffle one feature, how much does the model performance decrease?

If shuffling a feature causes a large drop in R2, that feature is important for the trained model.

In [ ]:
def get_permutation_importance_table(model, X_test, y_test, features):
    perm = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=30,
        random_state=42,
        scoring='r2',
        n_jobs=1,
    )

    return pd.DataFrame(
        {
            'feature': features,
            'permutation_mean': perm.importances_mean,
            'permutation_std': perm.importances_std,
        }
    ).sort_values('permutation_mean', ascending=False)


full_perm_importance = get_permutation_importance_table(
    full_rf_results['model'],
    full_rf_results['X_test'],
    full_rf_results['y_test'],
    full_features,
)

reduced_perm_importance = get_permutation_importance_table(
    reduced_rf_results['model'],
    reduced_rf_results['X_test'],
    reduced_rf_results['y_test'],
    reduced_features,
)

full_perm_importance.to_csv(RESULTS_DIR / 'notebook5_permutation_importance_full.csv', index=False)
reduced_perm_importance.to_csv(RESULTS_DIR / 'notebook5_permutation_importance_reduced.csv', index=False)

full_perm_importance

### Mathematical meaning of permutation importance

For feature $j$, the importance is the decrease in model score after shuffling that feature:

$$
I_j = S_{\mathrm{base}} - S_{\mathrm{perm},j}.
$$

If shuffling feature $j$ strongly damages prediction quality, then $I_j$ becomes large.


### Explanation of the code cell above

This cell computes permutation importance.

Permutation importance works as follows:

1. Train the model normally.
2. Shuffle one feature column in the test set.
3. Recalculate model performance.
4. Measure how much the performance decreased.

If shuffling a feature strongly damages performance, that feature is important for prediction.

This method is closer to an experimental sensitivity test than built-in tree importance.


In [ ]:
reduced_perm_importance

### Explanation of the code cell above

This cell displays the permutation-importance table for the reduced model.

The columns show:

- average importance across repeated shuffles,
- standard deviation across those repeats.

The standard deviation helps you judge whether the importance estimate is stable.


## Plot Permutation Importance

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(
    full_perm_importance['feature'][::-1],
    full_perm_importance['permutation_mean'][::-1],
    xerr=full_perm_importance['permutation_std'][::-1],
)
plt.xlabel('Permutation importance decrease in R2')
plt.ylabel('Feature')
plt.title('Permutation importance - full-feature model')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook5_permutation_importance_full.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.barh(
    reduced_perm_importance['feature'][::-1],
    reduced_perm_importance['permutation_mean'][::-1],
    xerr=reduced_perm_importance['permutation_std'][::-1],
)
plt.xlabel('Permutation importance decrease in R2')
plt.ylabel('Feature')
plt.title('Permutation importance - reduced-feature model')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook5_permutation_importance_reduced.png', dpi=200)
plt.show()

### Mathematical meaning of permutation importance

For feature $j$, the importance is the decrease in model score after shuffling that feature:

$$
I_j = S_{\mathrm{base}} - S_{\mathrm{perm},j}.
$$

If shuffling feature $j$ strongly damages prediction quality, then $I_j$ becomes large.


### Explanation of the code cell above

This cell plots permutation importance for both the full and reduced models.

The horizontal bars show the average performance decrease after shuffling each feature. The error bars show variability across repeated permutations.

A feature with a large positive value is important because the model depends on it for accurate prediction.


## Compare RF Importance and Permutation Importance

Random Forest importance and permutation importance measure feature relevance differently.

Random Forest importance measures how much a feature reduces impurity inside the model.

Permutation importance measures how much the test-set score decreases when a feature is randomly shuffled.

If both methods identify the same important features, the interpretation becomes more reliable.

In [ ]:
full_importance_comparison = full_rf_importance.merge(
    full_perm_importance,
    on='feature',
    how='inner',
)

reduced_importance_comparison = reduced_rf_importance.merge(
    reduced_perm_importance,
    on='feature',
    how='inner',
)

full_importance_comparison.to_csv(RESULTS_DIR / 'notebook5_importance_comparison_full.csv', index=False)
reduced_importance_comparison.to_csv(RESULTS_DIR / 'notebook5_importance_comparison_reduced.csv', index=False)

full_importance_comparison

### Explanation of the code cell above

This cell combines Random Forest importance and permutation importance into comparison tables.

This is useful because the two methods do not always agree.

If a feature is important in both methods, the interpretation is stronger. If the methods disagree, you should be more cautious and investigate further.


In [ ]:
reduced_importance_comparison

### Explanation of the code cell above

This cell displays the reduced-feature importance comparison.

This table is useful for writing the scientific interpretation because it places two importance methods side by side for the simpler model.


## Optional SHAP Analysis

SHAP analysis is optional. If available, it provides a more detailed view of how each feature contributes to individual predictions. If SHAP is not installed, Random Forest importance and permutation importance are sufficient for this proof-of-concept version.

In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

SHAP_AVAILABLE

### Explanation of the code cell above

This cell checks whether the optional `shap` package is installed.

SHAP is an advanced interpretability method that explains how each feature contributes to individual predictions.

The notebook is written safely: if SHAP is not installed, the rest of the notebook still works.


In [ ]:
if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(reduced_rf_results['model'])
    shap_values = explainer.shap_values(reduced_rf_results['X_test'])

    shap.summary_plot(
        shap_values,
        reduced_rf_results['X_test'],
        show=False,
    )

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / 'notebook5_shap_summary_reduced.png', dpi=200)
    plt.show()

### Explanation of the code cell above

This cell runs SHAP analysis only if SHAP is available.

The SHAP summary plot shows both:

- which features are important,
- whether high or low feature values push predictions upward or downward.

This is optional because SHAP may not be installed in every environment.


## Scientific Interpretation

The full-feature model is expected to be dominated by `defect_density` and `mean_ipr`. This is expected because the transport descriptor is explicitly defined as:

```text
T_score = 1 / (1 + mean_IPR + defect_density)
```

Therefore, the full-feature importance result validates that the model is learning the designed mathematical proxy.

The reduced-feature model is more scientifically informative because it removes `defect_density` and `mean_ipr`. In this setting, important features such as `clustering_index`, `max_ipr`, `mean_defect_distance`, and `energy_bandwidth` indicate that indirect defect-configuration and spectral descriptors carry information about the transport-related score.

However, feature importance should not be interpreted as proof of physical causality. It only tells us which variables the trained model used most strongly for prediction.

A safe interpretation is:

> The interpretability analysis suggests that defect clustering, maximum localization, and spectral bandwidth contain useful indirect information for predicting the designed transport-related descriptor.

## Conclusion

In this notebook, we performed interpretability analysis for the machine-learning models trained on the hydrogen-defect dataset.

For the full-feature model, both Random Forest importance and permutation importance are expected to emphasize `defect_density` and `mean_ipr`, because these variables directly define the transport descriptor.

For the reduced-feature model, the most important features are expected to include `clustering_index`, `max_ipr`, `mean_defect_distance`, and `energy_bandwidth`. This indicates that defect arrangement, strong local localization, and spectral structure contain indirect information about the transport-related score.

These results support a scientifically safe interpretation: the machine-learning models do not discover a new transport law, but they can reproduce and partially infer a designed physics-informed transport proxy from interpretable physical descriptors.

The next step is Notebook 6, where we use optimization methods to search for defect configurations that maximize the transport-related descriptor.

## Key learning takeaway

After running this notebook, focus on writing one or two sentences in your own words:

- What physical quantity was computed?
- Which feature or parameter changed?
- How did the result affect localization or transport?

This habit will help you turn code execution into scientific understanding.


## Final Formula Reminder

The core logic of this notebook can be summarized as

$$
\text{defect configuration} \rightarrow H \rightarrow \{E_n,\mathbf{c}^{(n)}\} \rightarrow \text{features} \rightarrow \text{prediction or optimization}.
$$

This is the main physics-informed workflow: we start from a material-inspired defect model, convert it into spectral and localization descriptors, and then use those descriptors for machine learning, interpretation, or optimization.
